# Direct Preference Optimization (DPO) & Odds Ratio Preference Optimization (ORPO) on Qwen3 with MaxText

This notebook demonstrates the end-to-end workflow of fine-tuning a model using Direct Preference Optimization (DPO) or Odds Ratio Preference Optimization (ORPO) with MaxText.

Both DPO and ORPO are stable and computationally efficient methods for aligning large language models to human preferences, bypassing the need for fitting a separate reward model or using complex reinforcement learning loops (like PPO).

### Workflow Overview:
1.  **Setup and Checkpoint Conversion**: Download a pre-trained SFT model (Qwen3-0.6B-Instruct) and convert it to MaxText format.
2.  **Baseline Evaluation**: Evaluate the SFT model's instruction-following capabilities on the `IFEval` benchmark before DPO training.
3.  **DPO Training**: Fine-tune the model on a preference dataset (`distilabel-intel-orca-dpo-pairs`) using the Tunix DPOTrainer integrated with MaxText.
4.  **Post-DPO Evaluation**: Evaluate the aligned model on `IFEval` to verify behavioral improvement.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AI-Hypercomputer/maxtext/blob/main/src/maxtext/examples/dpo_qwen3_demo.ipynb)

## Prerequisites

Before running this notebook, ensure you have access to a TPU (v4 or newer is recommended). 
If you are running in Google Colab, make sure to select a TPU v2-8 or TPU v4-8 runtime.

In [8]:
try:
  import google.colab
  print("Running the notebook on Google Colab")
  IN_COLAB = True
except ImportError:
  print("Running the notebook on local/remote VM")
  IN_COLAB = False

Running the notebook on local/remote VM


## Installation: MaxText and Post-training Dependencies

We need to install MaxText and additional dependencies for post-training (Tunix) and evaluation (lm-evaluation-harness).

In [9]:
if IN_COLAB:
    # Clone the MaxText repository
    !git clone https://github.com/AI-Hypercomputer/maxtext.git
    %cd maxtext

    # Install uv, a fast Python package installer
    !pip install uv

    # Install MaxText and post-training dependencies
    import os
    os.environ["UV_TORCH_BACKEND"]="cpu"
    !uv pip install -e .[tpu-post-train] --resolution=lowest
    !install_tpu_post_train_extra_deps

# Install evaluation dependencies
try:
    import lm_eval
    import nest_asyncio
    import langdetect
except ImportError:
    if IN_COLAB:
        !uv pip install "lm_eval[api]" nest_asyncio langdetect
    else:
        %pip install "lm_eval[api]" nest_asyncio langdetect

## Environment Setup & Imports

Initialize JAX distributed system (if running on multi-host) and import necessary libraries.

In [10]:
import os
import sys
import subprocess
import glob
import json
import logging
from etils import epath

# Configure logging to see evaluation progress
logging.basicConfig(level=logging.INFO, format="%(levelname)s:%(name)s:%(message)s")

# Add src to path if not running from root
sys.path.append(os.path.abspath("src"))

from maxtext.utils.globals import MAXTEXT_PKG_DIR

print("Environment setup and imports completed successfully.")

Environment setup and imports completed successfully.


In [11]:
if IN_COLAB:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Ensure you have logged in via `huggingface-cli login` or set HF_TOKEN env var.")

Ensure you have logged in via `huggingface-cli login` or set HF_TOKEN env var.


## Configurations

Define the model, dataset, and paths for training and evaluation.
We will use Qwen3-0.6B-Instruct as our SFT baseline, and align it using DPO on the `distilabel-intel-orca-dpo-pairs` dataset.

In [12]:
MODEL_NAME = "qwen3-0.6b"
TOKENIZER_PATH = "Qwen/Qwen3-0.6B"
HF_PATH = "Qwen/Qwen3-0.6B"
NUM_DPO_TRAINING_STEPS = 500
PER_DEVICE_BATCH_SIZE = 4
DPO_LEARNING_RATE = 3e-6
TENSOR_PARALLEL_SIZE = 4
EVAL_MAX_NUM_SEQS = 64
DPO_ALGORITHM = "dpo"  # Alignment algorithm to use. Can be set to "dpo" or "orpo"

# Local directories
BASE_DIR = "/tmp/dpo_demo"
MODEL_CHECKPOINT_PATH = f"{BASE_DIR}/sft_baseline_checkpoint"
DPO_OUTPUT_DIR = f"{BASE_DIR}/dpo_output"
RUN_NAME = "dpo_qwen3_demo"

# Create directories
os.makedirs(BASE_DIR, exist_ok=True)

## Download and Convert SFT Checkpoint

We download the pre-trained SFT model from Hugging Face and convert it to MaxText format.
This converted checkpoint will serve as the starting point (and reference model) for DPO.

In [13]:
if not epath.Path(MODEL_CHECKPOINT_PATH).exists():
    # Install torch for the conversion script
    print("Installing torch for conversion...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "torch", "--index-url", "https://download.pytorch.org/whl/cpu"
        ],
        check=True
    )

    print("Converting checkpoint from HuggingFace...")
    env = os.environ.copy()
    env["JAX_PLATFORMS"] = "cpu"

    subprocess.run(
        [
            sys.executable,
            "-m", "maxtext.checkpoint_conversion.to_maxtext",
            "src/maxtext/configs/base.yml",
            f"model_name={MODEL_NAME}",
            f"base_output_directory={MODEL_CHECKPOINT_PATH}",
            f"--hf_model_path={HF_PATH}",
            "use_multimodal=false",
            "scan_layers=true",
            "skip_jax_distributed_system=True",
        ],
        check=True,
        env=env
    )

    CONVERTED_SFT_CKPT = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
    print(f"Checkpoint converted to {CONVERTED_SFT_CKPT}")
else:
    CONVERTED_SFT_CKPT = os.path.join(MODEL_CHECKPOINT_PATH, "0/items")
    print(f"Model checkpoint already exists at {CONVERTED_SFT_CKPT}")

Installing torch for conversion...
Looking in indexes: https://download.pytorch.org/whl/cpu
Converting checkpoint from HuggingFace...


/home/igorts_google_com/git/maxtext/maxtext_venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
2026-07-06 16:46:19.958761: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-06 16:46:20.023898: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-06 16:46:21.350400: I tensorflow/core/util/port.cc:153]

Checkpoint converted to /tmp/dpo_demo/sft_baseline_checkpoint/0/items


## Step 1: Evaluation before DPO (SFT Baseline)

We evaluate the converted SFT checkpoint on the `IFEval` task to establish a baseline.
We use the unified evaluation runner with `--runner lm_eval`.

In [14]:
# Run Pre-DPO Evaluation on IFEval (via isolated subprocess)
print("Running Pre-DPO Evaluation on IFEval (via isolated subprocess)...")

eval_cmd = [
    sys.executable,
    "-m", "maxtext.eval.runner.run",
    "--runner", "lm_eval",
    f"--checkpoint_path={CONVERTED_SFT_CKPT}",
    f"--model_name={MODEL_NAME}",
    f"--hf_path={HF_PATH}",
    "--tasks", "ifeval",
    f"--base_output_directory={BASE_DIR}",
    "--run_name=sft_baseline",
    "--max_model_len=1024",
    f"--tensor_parallel_size={TENSOR_PARALLEL_SIZE}",
    f"--max_num_seqs={EVAL_MAX_NUM_SEQS}",
    "--hbm_memory_utilization=0.95",
    "--apply_chat_template",
]

subprocess.run(eval_cmd, check=True)

# Locate and read the newest generated results JSON file
results_dir = f"{BASE_DIR}/sft_baseline/eval_results"
json_files = glob.glob(os.path.join(results_dir, "*.json"))
newest_results_file = max(json_files, key=os.path.getmtime)

with open(newest_results_file, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

run_results_pre = {
    "results_file": newest_results_file,
    "scores": eval_data.get("scores", {})
}

print("Pre-DPO Evaluation Completed.")
print(f"Results saved to: {run_results_pre.get('results_file')}")
print(f"Scores: {run_results_pre.get('scores')}")

Running Pre-DPO Evaluation on IFEval (via isolated subprocess)...


/home/igorts_google_com/git/maxtext/maxtext_venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
2026-07-06 16:46:45.606421: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-06 16:46:45.671351: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-06 16:46:46.980185: I tensorflow/core/util/port.cc:153]

INFO 07-06 16:46:51 [__init__.py:59] TPU info: node_name=igorts-vm | tpu_type=v5p-8 | worker_id=0 | num_chips=4 | num_cores_per_chip=2


E0000 00:00:1783356412.531515 1054890 descriptor_database.cc:633] File already exists in database: google/protobuf/timestamp.proto
F0000 00:00:1783356412.531597 1054890 descriptor.cc:2236] Check failed: GeneratedDatabase()->Add(encoded_file_descriptor, size) 
*** Check failure stack trace: ***
    @     0x7f726c0a2fe4  absl::lts_20250127::log_internal::LogMessage::SendToLog()
    @     0x7f726c0a2976  absl::lts_20250127::log_internal::LogMessage::Flush()
    @     0x7f726c0a3539  absl::lts_20250127::log_internal::LogMessageFatal::~LogMessageFatal()
    @     0x7f726bf955cb  google::protobuf::DescriptorPool::InternalAddGeneratedFile()
    @     0x7f726c00f308  google::protobuf::internal::AddDescriptors()
    @     0x7f726c00f2fa  google::protobuf::internal::AddDescriptors()
    @     0x7f7432350b9f  __static_initialization_and_destruction_0()
    @     0x7f7432350bd2  _GLOBAL__sub_I.00102_tpu_metric_service.pb.cc
    @     0x7f74441c447e  (unknown)


Check failed with unknown exit code: -6.
INFO 07-06 16:46:53 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 07-06 16:46:53 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.
WARNING 07-06 16:46:53 [interface.py:240] Failed to import from vllm._C: ModuleNotFoundError("No module named 'vllm._C'")
INFO 07-06 16:46:53 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 07-06 16:46:53 [nixl_utils.py:34] NIXL is not available
WARNING 07-06 16:46:53 [nixl_utils.py:44] NIXL agent config is not available
WARNING 07-06 16:46:53 [tpu_platform.py:317] Pin memory is not supported on TPU.
INFO 07-06 16:46:53 [__init__.py:110] Registered model loader `<class 'tpu_inference.models.jax.utils.weight_utils.JaxDummyModelLoader'>` with load format `jax_dummy`
INFO 07-06 16:46:53 [attention_interf

...

WARNING 07-06 16:48:39 [tpu_runner.py:808] Should not schedule a request that does nothing!



Rendering prompts: 100%|██████████| 64/64 [00:00<00:00, 10822.26it/s]

Rendering prompts: 100%|██████████| 64/64 [00:00<00:00, 10751.61it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 07-06 16:48:46 [tpu_runner.py:808] Should not schedule a request that does nothing!



Rendering prompts: 100%|██████████| 64/64 [00:00<00:00, 10527.29it/s]

Rendering prompts: 100%|██████████| 29/29 [00:00<00:00, 10039.19it/s]

Processed prompts:   0%|          | 0/29 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]W0706 16:49:03.278995 1056560 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.

Requesting API: 100%|██████████| 541/541 [01:16<00:00,  7.03it/s]


Pre-DPO Evaluation Completed.
Results saved to: /tmp/dpo_demo/sft_baseline/eval_results/ifeval_qwen3-0.6b_20260706T164912Z.json
Scores: {'ifeval_prompt_level_strict_acc': 31.61, 'ifeval_inst_level_strict_acc': 44.6, 'ifeval_prompt_level_loose_acc': 33.27, 'ifeval_inst_level_loose_acc': 46.16}


## Step 2: Direct Preference Optimization (DPO) Training

Now we run DPO training to align the model. We use the `train_dpo` script programmatically.
We configure it to run for a small number of steps for demonstration purposes.

In [15]:
# Run DPO training via isolated subprocess
print("Starting DPO training (via isolated subprocess)...")
subprocess.run(
    [
        sys.executable,
        "-m", "maxtext.trainers.post_train.dpo.train_dpo",
        f"{MAXTEXT_PKG_DIR}/configs/post_train/dpo.yml",
        f"load_parameters_path={CONVERTED_SFT_CKPT}",
        f"model_name={MODEL_NAME}",
        f"base_output_directory={DPO_OUTPUT_DIR}",
        f"run_name={RUN_NAME}",
        f"tokenizer_path={TOKENIZER_PATH}",
        "dataset_type=hf",
        "hf_path=argilla/distilabel-intel-orca-dpo-pairs",
        "train_split=train",
        "hf_eval_split=train",
        "train_data_columns=['input', 'chosen', 'rejected']",
        f"per_device_batch_size={PER_DEVICE_BATCH_SIZE}",
        "max_target_length=1024",
        f"steps={NUM_DPO_TRAINING_STEPS}",  # Running for steps for the demo
        f"eval_interval={NUM_DPO_TRAINING_STEPS}",
        f"learning_rate={DPO_LEARNING_RATE}",
        f"dpo.algo={DPO_ALGORITHM}",
        "weight_dtype=bfloat16",
        "dtype=bfloat16",
        "scan_layers=True",
        "enable_checkpointing=True",
        "async_checkpointing=False",
    ],
    check=True,
)
print("DPO training completed!")

# Path to the trained DPO checkpoint
DPO_CHECKPOINT = os.path.join(DPO_OUTPUT_DIR, RUN_NAME, "checkpoints", str(NUM_DPO_TRAINING_STEPS))
print(f"DPO checkpoint saved at: {DPO_CHECKPOINT}")

Starting DPO training (via isolated subprocess)...


/home/igorts_google_com/git/maxtext/maxtext_venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
2026-07-06 16:49:22.881989: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-06 16:49:22.946937: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-06 16:49:24.269861: I tensorflow/core/util/port.cc:153]

Per train step:
 Total TFLOPs: 21.45 
 split as 68.27% learnable weight flops and 8.97% attention flops


/home/igorts_google_com/git/maxtext/maxtext_venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
2026-07-06 16:50:05.318709: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-06 16:50:05.383159: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-06 16:50:06.693052: I tensorflow/core/util/port.cc:153]

DPO training completed!
DPO checkpoint saved at: /tmp/dpo_demo/dpo_output/dpo_qwen3_demo/checkpoints/500


## Step 3: Evaluation after DPO

We evaluate the DPO-aligned model on `IFEval` to see the improvement in instruction-following performance.

In [16]:
# Run Post-DPO Evaluation on IFEval (via isolated subprocess)
print("Running Post-DPO Evaluation on IFEval (via isolated subprocess)...")

eval_cmd = [
    sys.executable,
    "-m", "maxtext.eval.runner.run",
    "--runner", "lm_eval",
    f"--checkpoint_path={DPO_CHECKPOINT}",
    f"--model_name={MODEL_NAME}",
    f"--hf_path={HF_PATH}",
    "--tasks", "ifeval",
    f"--base_output_directory={BASE_DIR}",
    "--run_name=dpo_aligned",
    "--max_model_len=1024",
    f"--tensor_parallel_size={TENSOR_PARALLEL_SIZE}",
    f"--max_num_seqs={EVAL_MAX_NUM_SEQS}",
    "--hbm_memory_utilization=0.95",
    "--apply_chat_template",
]

subprocess.run(eval_cmd, check=True)

# Locate and read the newest generated results JSON file
results_dir = f"{BASE_DIR}/dpo_aligned/eval_results"
json_files = glob.glob(os.path.join(results_dir, "*.json"))
newest_results_file = max(json_files, key=os.path.getmtime)

with open(newest_results_file, "r", encoding="utf-8") as f:
    eval_data = json.load(f)

run_results_post = {
    "results_file": newest_results_file,
    "scores": eval_data.get("scores", {})
}

print("Post-DPO Evaluation Completed.")
print(f"Results saved to: {run_results_post.get('results_file')}")
print(f"Scores: {run_results_post.get('scores')}")

Running Post-DPO Evaluation on IFEval (via isolated subprocess)...


/home/igorts_google_com/git/maxtext/maxtext_venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(
2026-07-06 16:53:09.031280: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-06 16:53:09.096707: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-06 16:53:10.398862: I tensorflow/core/util/port.cc:153]

INFO 07-06 16:53:15 [__init__.py:59] TPU info: node_name=igorts-vm | tpu_type=v5p-8 | worker_id=0 | num_chips=4 | num_cores_per_chip=2


E0000 00:00:1783356795.956341 1061192 descriptor_database.cc:633] File already exists in database: google/protobuf/timestamp.proto
F0000 00:00:1783356795.956422 1061192 descriptor.cc:2236] Check failed: GeneratedDatabase()->Add(encoded_file_descriptor, size) 
*** Check failure stack trace: ***
    @     0x7f9fbfca2fe4  absl::lts_20250127::log_internal::LogMessage::SendToLog()
    @     0x7f9fbfca2976  absl::lts_20250127::log_internal::LogMessage::Flush()
    @     0x7f9fbfca3539  absl::lts_20250127::log_internal::LogMessageFatal::~LogMessageFatal()
    @     0x7f9fbfb955cb  google::protobuf::DescriptorPool::InternalAddGeneratedFile()
    @     0x7f9fbfc0f308  google::protobuf::internal::AddDescriptors()
    @     0x7f9fbfc0f2fa  google::protobuf::internal::AddDescriptors()
    @     0x7fa185f50b9f  __static_initialization_and_destruction_0()
    @     0x7fa185f50bd2  _GLOBAL__sub_I.00102_tpu_metric_service.pb.cc
    @     0x7fa197d8747e  (unknown)


Check failed with unknown exit code: -6.
INFO 07-06 16:53:16 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 07-06 16:53:16 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.
WARNING 07-06 16:53:16 [interface.py:240] Failed to import from vllm._C: ModuleNotFoundError("No module named 'vllm._C'")
INFO 07-06 16:53:16 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 07-06 16:53:16 [nixl_utils.py:34] NIXL is not available
WARNING 07-06 16:53:16 [nixl_utils.py:44] NIXL agent config is not available
WARNING 07-06 16:53:16 [tpu_platform.py:317] Pin memory is not supported on TPU.
INFO 07-06 16:53:16 [__init__.py:110] Registered model loader `<class 'tpu_inference.models.jax.utils.weight_utils.JaxDummyModelLoader'>` with load format `jax_dummy`
INFO 07-06 16:53:16 [attention_interf

...

WARNING 07-06 16:55:03 [tpu_runner.py:808] Should not schedule a request that does nothing!



Rendering prompts: 100%|██████████| 64/64 [00:00<00:00, 10493.96it/s]

Rendering prompts: 100%|██████████| 64/64 [00:00<00:00, 10392.39it/s]

Processed prompts:   0%|          | 0/64 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 07-06 16:55:10 [tpu_runner.py:808] Should not schedule a request that does nothing!



Rendering prompts: 100%|██████████| 64/64 [00:00<00:00, 10494.78it/s]

Rendering prompts: 100%|██████████| 29/29 [00:00<00:00, 8824.35it/s]

Processed prompts:   0%|          | 0/29 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]W0706 16:55:27.176337 1062849 pjrt_executable.cc:642] Assume version compatibility. PjRt-IFRT does not track XLA executable versions.

Requesting API: 100%|██████████| 541/541 [01:17<00:00,  7.02it/s]


Post-DPO Evaluation Completed.
Results saved to: /tmp/dpo_demo/dpo_aligned/eval_results/ifeval_qwen3-0.6b_20260706T165536Z.json
Scores: {'ifeval_prompt_level_strict_acc': 31.61, 'ifeval_inst_level_strict_acc': 44.6, 'ifeval_prompt_level_loose_acc': 33.27, 'ifeval_inst_level_loose_acc': 46.04}


## Summary

In this notebook, you have:
1. Converted a Hugging Face SFT model to MaxText format.
2. Evaluated the baseline SFT model on the IFEval benchmark.
3. Aligned the model using Direct Preference Optimization (DPO) with MaxText and Tunix.
4. Evaluated the DPO-aligned model to verify improvement.

For more details on DPO configurations and scaling, refer to the [MaxText Post-Training Documentation](https://github.com/AI-Hypercomputer/maxtext/blob/main/src/maxtext/trainers/post_train/README.md).